In [ ]:
%pip install zipline-ai zipline-pyspark --quiet

In [ ]:
# spark, dbutils, and display are provided by the Databricks runtime — no SparkSession needed.
#
# The Chronon batch JAR must be installed as a cluster library before running this notebook.
# In the Databricks UI: Compute → <your cluster> → Libraries → Install new → JAR
#   s3://zipline-artifacts-canary/release/latest/jars/cloud_aws_deploy.jar

import sys
import os

canary_dir = "/dbfs/FileStore/canary"
sys.path.append(canary_dir)
os.environ["CHRONON_ROOT"] = canary_dir

In [ ]:
# Run this cell once per session to create the date-range widgets in the toolbar above.
from ai.chronon.pyspark.databricks import DatabricksStagingQuery

from staging_queries.aws_databricks.exports import dim_listings

sq_executor = DatabricksStagingQuery(
    dim_listings,
    spark,
    dbutils=dbutils,
    display_fn=display,
)
sq_executor.setup_widgets(default_end_date="2026-01-01", default_start_date="2025-12-01")

In [ ]:
# Reads the widget values and runs the backfill. The result is displayed inline via display().
end_date, start_date = sq_executor.get_widget_dates()
sq_executor.run(end_date, start_date=start_date)

In [ ]:
from ai.chronon.pyspark.databricks import DatabricksGroupBy

from group_bys.aws.dim_listings import v1 as dim_listings_gb

DatabricksGroupBy(dim_listings_gb, spark, display_fn=display).run(
    end_date="2026-01-01",
    start_date="2025-12-01",
)

In [ ]:
from ai.chronon.pyspark.databricks import DatabricksJoin

from joins.aws.training_set import v1_dev

DatabricksJoin(v1_dev, spark, display_fn=display).run(
    end_date="2026-01-01",
    start_date="2025-12-01",
)